# 02 · NE Brazil dataset (PLAN_BRAZIL phase 3)
Builds `$DATA_ROOT/brazil/` in the layout `Source/` expects: INMET stations take the place of MADIS, ERA5 stays the gridded input.
Every step skips work that is already done, so rerun the whole notebook after an interruption. CPU runtime is enough.

| step | script | output under `$DATA_ROOT/brazil/` |
|---|---|---|
| 1 | `build_region.py` | `Shapefiles/Regions/ne_brazil_buffered.shp` (CE, RN, PB, PE; IBGE 2020) |
| 2 | `download_inmet.py` | `inmet/raw/{year}/` from `inmet/zips/{year}.zip` |
| 3 | `inmet_to_localized.py` | `madis/stations/*.shp`, `madis/processed/Meta-{start}-{end}/*.nc`, `inmet/station_counts.csv` |
| 4 | `download_era5.py` | `ERA5/NEBrazil/surface_{year}_{month}.nc` |
| 5 | `validate_dataset.py` | `report/brazil_data_report.md` + figures (copy to `docs/`) |

In [ ]:
# Setup. Colab: clones/updates the repo, installs requirements, mounts Drive.
# PC: start Jupyter anywhere inside the repo and set the DATA_ROOT environment variable first.
import os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    repo = Path('/content/LocalizedWeatherBR')
    if repo.exists():
        subprocess.run(['git', '-C', str(repo), 'pull', '-q'], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', 'https://github.com/joancsz/LocalizedWeatherBR.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/LocalizedWeather'
else:
    os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PLAN_BRAZIL.md').exists()))
    DATA_ROOT = os.environ.get('DATA_ROOT') or sys.exit('set DATA_ROOT (folder holding neus/ and brazil/) before starting Jupyter')

os.environ['DATA_ROOT'] = DATA_ROOT  # configs read ${DATA_ROOT}
PY = sys.executable                   # !{PY} runs the scripts with this kernel's Python on Colab and Windows alike
print('repo:', os.getcwd(), '| DATA_ROOT:', DATA_ROOT)

In [ ]:
YEARS = '2019 2020 2021 2022 2023'   # must match start_year..end_year in configs/ne_brazil*.yaml
CONTROL_RATIO = 0.6   # 2019-2023 keeps 26/23/15/6 stations at 0.6/0.7/0.8/0.9; 0.6 is the only one >= 25

## 1 · Region

In [ ]:
!{PY} scripts/brazil/build_region.py --data-root "{DATA_ROOT}"

## 2 · INMET archives
The INMET portal refuses scripted downloads from some networks (connection reset). If the download fails, get
`https://portal.inmet.gov.br/uploads/dadoshistoricos/{year}.zip` in a browser and put the files in
`$DATA_ROOT/brazil/inmet/zips/`; the script then only extracts them.

In [ ]:
!{PY} scripts/brazil/download_inmet.py --data-root "{DATA_ROOT}" --years {YEARS}

## 3 · INMET → LocalizedWeather files
Parses, quality-controls (ranges, dewpoint ≤ T + 0.5, 6 h flat lines) and converts to MADIS units; fails if the
diurnal temperature peak says the timestamps are not UTC. The log prints the retained-station table per control ratio:
the ratio was set to 0.6 because only 15 stations survive at 0.8 (see `docs/repro_log.md` P3). Changing it
later only rebuilds the filtered files, not the parse.

In [ ]:
!{PY} scripts/brazil/inmet_to_localized.py --data-root "{DATA_ROOT}" --years {YEARS} --control-ratio {CONTROL_RATIO}

## 4 · ERA5
Needs a Copernicus CDS key: put `.cdsapirc` in `$DATA_ROOT/` (on Drive for Colab), the repo root, or `~`.
The CDS queues each month for several minutes; 60 months take a while. A rerun re-requests only missing or broken months.

In [ ]:
!{PY} scripts/brazil/download_era5.py --data-root "{DATA_ROOT}" --years {YEARS}

## 5 · Validation report
Checks the schema and invariants I1–I5 on every file (including that the diurnal temperature peak of each station
agrees with ERA5 within ±1 h), and writes the report with a station map, availability heatmap and ERA5-vs-INMET ρ².
A failing check makes the cell exit non-zero; the report still lists what failed.

In [ ]:
!{PY} scripts/brazil/validate_dataset.py --data-root "{DATA_ROOT}" --years {YEARS} --control-ratio {CONTROL_RATIO}

In [ ]:
from IPython.display import Image, Markdown, display
report = Path(DATA_ROOT) / 'brazil' / 'report'
display(Markdown((report / 'brazil_data_report.md').read_text(encoding='utf-8')))
display(Image(report / 'stations_map.png'), Image(report / 'availability.png'))